In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("ProyectoVentas") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

print("Spark:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/16 15:31:33 WARN Utils: Your hostname, MacBook-Pro-de-MUSTAFA.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.143 instead (on interface en0)
26/08/16 15:31:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/mustafa/data_engineering/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/16 15:31:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.2.0


In [2]:
clientes = spark.read.csv(
    "../data/clientes.csv",
    header=True,
    inferSchema=True
)

productos = spark.read.csv(
    "../data/productos.csv",
    header=True,
    inferSchema=True
)

ventas = spark.read.csv(
    "../data/ventas.csv",
    header=True,
    inferSchema=True
)

print("Datos cargados correctamente")

Datos cargados correctamente


In [3]:
ventas_cliente = ventas.join(
    clientes,
    ventas.id_cliente == clientes.id_cliente,
    "inner"
)
ventas_completo = ventas_cliente.join(
    productos,
    ventas_cliente.id_producto == productos.id_producto,
    "inner"
)   


In [4]:
ventas_completo.select(
    "nombre",
    "ciudad",
    "producto",
    "categoria",
    "cantidad",
    "precio"
).show()

+------+---------+--------+-----------+--------+------+
|nombre|   ciudad|producto|  categoria|cantidad|precio|
+------+---------+--------+-----------+--------+------+
|   Ana|   Madrid|Portatil|Ordenadores|       1|   900|
|  Juan|Barcelona|   Raton| Accesorios|       2|    25|
| Pedro| Valencia| Teclado| Accesorios|       1|    50|
|   Ana|   Madrid| Monitor|  Monitores|       2|   200|
| Laura|  Sevilla|   Raton| Accesorios|       1|    25|
+------+---------+--------+-----------+--------+------+



In [7]:
v = ventas.alias("v")
c = clientes.alias("c")
p = productos.alias("p")

In [9]:
ventas.withColumn( "total",
F.col("cantidad") * F.col("precio")
)
ventas.select(
    "id_venta",
    "id_cliente",
    "id_producto",
    "cantidad",
    "precio",
    (F.col("cantidad") * F.col("precio")).alias("total")
).show()

+--------+----------+-----------+--------+------+-----+
|id_venta|id_cliente|id_producto|cantidad|precio|total|
+--------+----------+-----------+--------+------+-----+
|       1|         1|        101|       1|   900|  900|
|       2|         2|        102|       2|    25|   50|
|       3|         3|        103|       1|    50|   50|
|       4|         1|        104|       2|   200|  400|
|       5|         4|        102|       1|    25|   25|
+--------+----------+-----------+--------+------+-----+



In [12]:
ventas.withColumn(
    "tipoventa",
    F.when(F.col("cantidad")> 1, "varias unidades").otherwise("una unidad")).select(
    "id_venta",
    "id_cliente",
    "id_producto",
    "cantidad",
    "precio",
    "tipoventa"
).show()

+--------+----------+-----------+--------+------+---------------+
|id_venta|id_cliente|id_producto|cantidad|precio|      tipoventa|
+--------+----------+-----------+--------+------+---------------+
|       1|         1|        101|       1|   900|     una unidad|
|       2|         2|        102|       2|    25|varias unidades|
|       3|         3|        103|       1|    50|     una unidad|
|       4|         1|        104|       2|   200|varias unidades|
|       5|         4|        102|       1|    25|     una unidad|
+--------+----------+-----------+--------+------+---------------+



In [20]:
ventas_total=ventas.withColumn( "total",
F.col("cantidad") * F.col("precio")
)


In [22]:
ventas_total.groupBy("id_cliente").agg(
    F.sum("total").alias("total_gastado")
).show()

+----------+-------------+
|id_cliente|total_gastado|
+----------+-------------+
|         1|         1300|
|         3|           50|
|         4|           25|
|         2|           50|
+----------+-------------+



In [23]:
ventas_total.groupBy("id_cliente").agg(
    F.sum("total").alias("total_gastado"),
    F.count("*").alias("total_compras"),
    F.avg("total").alias("promedio_compra"),
    F.max("total").alias("compra_mas_alta")
).show()


+----------+-------------+-------------+---------------+---------------+
|id_cliente|total_gastado|total_compras|promedio_compra|compra_mas_alta|
+----------+-------------+-------------+---------------+---------------+
|         1|         1300|            2|          650.0|            900|
|         3|           50|            1|           50.0|             50|
|         4|           25|            1|           25.0|             25|
|         2|           50|            1|           50.0|             50|
+----------+-------------+-------------+---------------+---------------+



In [36]:
ventas.groupBy("id_cliente").agg(
    F.countDistinct("id_producto").alias("productos_diferentes")).show()

+----------+--------------------+
|id_cliente|productos_diferentes|
+----------+--------------------+
|         1|                   2|
|         3|                   1|
|         4|                   1|
|         2|                   1|
+----------+--------------------+



26/08/15 19:09:17 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 934398 ms exceeds timeout 120000 ms
26/08/15 19:09:17 WARN SparkContext: Killing executors is not supported by current scheduler.
26/08/15 19:09:19 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora

In [28]:
clientes_limpios = clientes.withColumn(
    "nombres_limpios",
    F.upper(F.trim(F.col("nombre"))))

In [29]:
clientes_limpios.show()

+----------+------+---------+---------------+
|id_cliente|nombre|   ciudad|nombres_limpios|
+----------+------+---------+---------------+
|         1|   Ana|   Madrid|            ANA|
|         2|  Juan|Barcelona|           JUAN|
|         3| Pedro| Valencia|          PEDRO|
|         4| Laura|  Sevilla|          LAURA|
+----------+------+---------+---------------+



In [32]:
clientes_mayusculas = clientes.withColumn(
    "nombre_mayusculas",
    F.upper(F.col("nombre")))
 

In [33]:
clientes_mayusculas.show()

+----------+------+---------+-----------------+
|id_cliente|nombre|   ciudad|nombre_mayusculas|
+----------+------+---------+-----------------+
|         1|   Ana|   Madrid|              ANA|
|         2|  Juan|Barcelona|             JUAN|
|         3| Pedro| Valencia|            PEDRO|
|         4| Laura|  Sevilla|            LAURA|
+----------+------+---------+-----------------+



In [34]:
cliente_nuevo = clientes.withColumn(
    "cliente_info",
    F.concat_ws(" - ", F.col("nombre"), F.col("ciudad")))

In [35]:
cliente_nuevo.show()

+----------+------+---------+----------------+
|id_cliente|nombre|   ciudad|    cliente_info|
+----------+------+---------+----------------+
|         1|   Ana|   Madrid|    Ana - Madrid|
|         2|  Juan|Barcelona|Juan - Barcelona|
|         3| Pedro| Valencia|Pedro - Valencia|
|         4| Laura|  Sevilla| Laura - Sevilla|
+----------+------+---------+----------------+



In [5]:
clientes_nonull = clientes.filter(
    F.col("ciudad").isNotNull()
)

In [6]:
clientes_nonull.show()

+----------+------+---------+
|id_cliente|nombre|   ciudad|
+----------+------+---------+
|         1|   Ana|   Madrid|
|         2|  Juan|Barcelona|
|         3| Pedro| Valencia|
|         4| Laura|  Sevilla|
+----------+------+---------+

